## 03. Process Addresses Data — objective

- Follow incremental ingestion from discovery to persisted results.
- Separate file discovery, parsing and schema tracking.

Historical results; no code executed.
- [Original inputs: circuitbox/addresses](../../../data/circuitbox/addresses/) — 2 files.

# 03. Process Addresses Data

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea06-delta-live-tables/03.%20Process%20Addresses%20Data.json)

## Learning notes — before source operation

**scd**: Change application uses keys and sequencing to resolve events. SCD type 1 replaces current values; type 2 retains validity intervals. Verify the actual configured type and event ordering.

## Process Addresses Data 
1. Ingest the data into the data lakehouse - bronze_addresses
1. Perform data quality checks and transform the data as required - silver_addresses_clean
1. Apply changes to the Addresses data (SCD Type 2) - silver_addresses

In [ ]:
import dlt
import pyspark.sql.functions as F

### 1. Ingest the data into the data lakehouse - bronze_addresses

![dea06-03-process-address-data-1.png](images/dea06-03-process-address-data-1.png)

In [ ]:
@dlt.table(
    name = "bronze_addresses",
    table_properties = {'quality' : 'bronze'},
    comment = "Raw addresses data ingested from the source system"
)
def create_bronze_addresses():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/circuitbox/landing/operational_data/addresses/")
            .select(
                "*",
                F.col("_metadata.file_path").alias("input_file_path"),
                F.current_timestamp().alias("ingest_timestamp")
            )
    )

### 2. Perform data quality checks and transform the data as required - silver_addresses_clean

In [ ]:
@dlt.table(
    name = "silver_addresses_clean",
    comment = "Cleaned addresses data",
    table_properties = {'quality' : 'silver'}
)
@dlt.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_address", "address_line_1 IS NOT NULL")
@dlt.expect("valid_postcode", "LENGTH(postcode) = 5")
def create_silver_addresses_clean():
    return (
        spark.readStream.table("LIVE.bronze_addresses")
            .select(
                "customer_id",
                "address_line_1",
                "city",
                "state",
                "postcode",
                F.col("created_date").cast("date")
            )
    )

### 3. Apply changes to the Addresses data (SCD Type 2) - silver_addresses

![dea06-03-process-address-data-2.png](images/dea06-03-process-address-data-2.png)

In [ ]:
dlt.create_streaming_table(
    name = "silver_addresses",
    comment = "SCD Type 2 addresses data",
    table_properties = {'quality' : 'silver'}
)

In [ ]:
dlt.apply_changes(
    target = "silver_addresses",
    source = "silver_addresses_clean",
    keys = ["customer_id"],
    sequence_by = "created_date",
    stored_as_scd_type = 2
)